# Multi-asset commodity futures portfolio

This notebook is a small, reviewable example of Screamer's first multi-asset
portfolio slice. It uses two synthetic but commodity-shaped futures series:
a liquid front contract and a slower deferred contract.

The important contract is deliberately narrow:

- targets are decided on the current close and execute at the next bar's open;
- the C++ OHLC target engine processes all asset columns in one call;
- contract multipliers convert price changes to dollar PnL;
- the returned arrays stay fixed-shape: (T, A, 4) per asset and (T, 10)
  for the aggregate portfolio;
- accounting is independent by instrument in this first slice; shared cash and
  cross-asset execution ordering are future work.

The data are seeded and offline so the notebook is safe to run locally.

In [ ]:
import sys
from pathlib import Path

# Make the notebook use the checkout when run from the repository or docs folder.
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / "screamer" / "__init__.py").is_file():
        sys.path.insert(0, str(candidate))
        break

import numpy as np
import matplotlib.pyplot as plt

from screamer import (
    RollingMean,
    portfolio_columns,
    portfolio_ohlc_target,
)

rng = np.random.default_rng(2026)
n = 360
returns = np.column_stack([
    0.004 * rng.standard_normal(n) + 0.0003,
    0.0025 * rng.standard_normal(n) + 0.0001,
])
returns[170:215, 0] += 0.0015
returns[240:285, 1] -= 0.0010

close = np.empty((n, 2), dtype=float)
close[0] = [75.0, 420.0]
close[1:] = close[0] * np.exp(np.cumsum(returns[1:], axis=0))
open_ = close * (1.0 + 0.0004 * rng.standard_normal((n, 2)))
high = np.maximum(open_, close) * (1.0 + 0.001 * rng.random((n, 2)))
low = np.minimum(open_, close) * (1.0 - 0.001 * rng.random((n, 2)))

fast = RollingMean(8)(close)
slow = RollingMean(30)(close)
targets = np.sign(np.nan_to_num(fast - slow, nan=0.0))
targets[:30] = 0.0

# A price point is $1,000 for the first contract and $50 for the second.
multipliers = np.array([1_000.0, 50.0])
margin_per_contract = np.array([8_000.0, 4_000.0])

result = portfolio_ohlc_target(
    targets,
    open_,
    high,
    low,
    close,
    multipliers=multipliers,
    margin_per_contract=margin_per_contract,
    taker_fee=0.0002,
    initial_cash=100_000.0,
)
print(result.asset_state.shape, result.portfolio_state.shape)
print(result.summary)

## Build a causal two-contract dataset

The two columns represent a front/deferred commodity pair. The prices are
synthetic; the point is the accounting and shape contract, not a market view.

In [ ]:
print("portfolio columns:")
for i, name in enumerate(portfolio_columns):
    print(f"{i:2d}  {name}")

print("\nlast portfolio row:")
print(dict(zip(portfolio_columns, result.portfolio_state[-1])))

## Inspect the fixed-shape result

asset_state[..., 0:4] is [equity, pnl, position, cost]. The aggregate columns
are named by portfolio_columns.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(result.portfolio_state[:, 0], label="portfolio equity")
axes[0].set_ylabel("equity ($)")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(result.portfolio_state[:, 3], label="gross exposure")
axes[1].plot(result.portfolio_state[:, 4], label="net exposure")
axes[1].set_ylabel("notional ($)")
axes[1].legend()
axes[1].grid(alpha=0.25)
fig.tight_layout()

## Causality check

A target from bar t cannot change the result on bar t; it is deferred to the
next open. This is the same causal rule as the single-asset engine.

In [ ]:
truncated = portfolio_ohlc_target(
    targets[:120],
    open_[:120],
    high[:120],
    low[:120],
    close[:120],
    multipliers=multipliers,
    margin_per_contract=margin_per_contract,
    taker_fee=0.0002,
    initial_cash=100_000.0,
)
np.testing.assert_allclose(
    result.asset_state[:120],
    truncated.asset_state,
    equal_nan=True,
)
np.testing.assert_allclose(
    result.portfolio_state[:120],
    truncated.portfolio_state,
    equal_nan=True,
)
print("batch truncation matches: no future bar changes an earlier result")

## Read the portfolio view

gross_exposure, net_exposure, margin_used, and drawdown are useful
portfolio-level diagnostics for a commodity trader. The first implementation
reports margin usage; it does not reserve or share cash between instruments.

In [ ]:
summary = result.summary
print("total PnL:", summary["total_pnl"])
print("max drawdown:", summary["max_drawdown"])
print("total cost:", summary["total_cost"])
print("turnover:", summary["turnover"])
print("number of trades:", summary["num_trades"])